# So s?nh thu?t to?n v? ablation ? 100 v?ng

Notebook ch?a ???c th?c thi. ??c b? 500 CSV c? s?n; kh?ng sinh d? li?u thay th?.

30 tr??ng h?p: 24 c?u h?nh WOA/PSO/GA, hai b?n lai ??y ?? v? b?n ablation c?a b?n lai. Xem `implementation/docs/NOTEBOOK_STUDY.md`.

**Ch? b?t ? ch?y sau khi ?? ch?t seed v? x?c nh?n ch?y.** Kh?ng c? s? li?u m?u gi? l?p.

In [ ]:
from pathlib import Path
import sys, json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "configs/notebook_study.json").exists())
sys.path.insert(0, str(ROOT / "implementation"))
from src.notebook_study import cases, plan, run_study, read_results, build_tables, GROUPS
CONFIG = json.loads((ROOT / "configs/notebook_study.json").read_text(encoding="utf-8"))
OUTPUT = ROOT / CONFIG["output"]
display(pd.DataFrame([CONFIG]))

## 1. B?ng thi?t k? v? kh?i l??ng ch?y
Danh s?ch seed hi?n ?? tr?ng, c?n ch?t tr??c khi ch?y. S? l??t ch?y = 500 ? 30 ? s? seed. Qu?n th? 25, 4 UAV v? 100 v?ng; c?ng seed/qu?n th? ng?u nhi?n n?n cho m?i tr??ng h?p. K-Means thay m?t c? th? c? ch? ??ch.

In [ ]:
design = pd.DataFrame(cases())
for group, names in GROUPS.items():
    display(Markdown(f"### {group}"))
    display(design.set_index("case").loc[names].reset_index().fillna("?"))
files, variants, total = plan(ROOT, CONFIG)
print(f"{len(files)} dataset; {len(variants)} tr??ng h?p; {len(CONFIG['seeds'])} seed; {total} l??t ch?y")

## 2. Ch?y th? nghi?m ? m?c ??nh T?T
Ch?t danh s?ch seed trong `configs/notebook_study.json` r?i n?p l?i c?u h?nh. Ch? ??t `CONFIRM_RUN = True` khi ???c y?u c?u ch?y. Ch?y tu?n t? ?? tr?nh l?i t?o process tr?n Windows.

M?i task ???c l?u JSON nguy?n t?; ch?y l?i c?ng c?u h?nh v? m? ngu?n s? b? qua task ?? ho?n th?nh. Kh?ng d?ng c?ng th? m?c cho hai kernel. N?u kernel b? kill, ki?m tra ti?n tr?nh ?? d?ng tr??c khi x? l? file kh?a `.running`.

In [ ]:
CONFIRM_RUN = False
if CONFIRM_RUN:
    OUTPUT = run_study(ROOT, CONFIG, confirmed=True)
else:
    print("Ch?a ch?y. C?n ch?t seed v? x?c nh?n r? tr??c khi b?t ? n?y.")

## 3. B?ng k?t qu? th?c t?
Ch? t?ng h?p khi ?? t?t c? task gh?p c?p. Ch?y c?c ? d??i sau khi c? k?t qu?; kh?ng g?i optimizer. Fitness l? t?t nh?t ?? ??nh gi? ??n h?t v?ng, k? c? ?ng vi?n kh?ng ???c gi? trong qu?n th?. SD qua seed ???c t?nh ri?ng t?ng dataset r?i l?y trung b?nh. FE ch? ??m ??nh gi? trong t?i ?u, kh?ng ??m t?nh ch? s? b?o c?o.

In [ ]:
tables = None
if (OUTPUT / "completion.json").exists():
    tables = build_tables(OUTPUT)
    for name, table in tables.items():
        display(Markdown(f"### {name}"))
        display(table)
else:
    print("Ch?a c? l?n ch?y ho?n t?t; kh?ng t?o b?ng k?t qu? ho?c k?t lu?n.")

## 4. ???ng h?i t? t?ng nh?m
Trung v? v? d?i ph?n v? 25?75% tr?n c?c dataset/seed; d?i n?y kh?ng ph?i kho?ng tin c?y. V?ng 0 l? sau kh?i t?o, g?m OBL kh?i t?o n?u b?t. C?c m?c 35/80 l? checkpoint c?a l?n ch?y 100 v?ng.

In [ ]:
if tables is not None:
    figure_dir = OUTPUT / "figures"
    figure_dir.mkdir(exist_ok=True)
    for group, names in GROUPS.items():
        fig, ax = plt.subplots(figsize=(10, 5))
        for name in names:
            g = tables["convergence"].query("case == @name")
            ax.plot(g.iteration, g["median"], label=name)
            ax.fill_between(g.iteration, g.q25, g.q75, alpha=.10)
        ax.set(xlabel="V?ng l?p", ylabel="Fitness t?t nh?t", title=group)
        ax.grid(alpha=.25); ax.legend(fontsize=8); fig.tight_layout()
        fig.savefig(figure_dir / f"convergence_{group}.png", dpi=180)
        plt.show()

## 5. K?t qu? theo quy m? ng??i d?ng
M?i ?i?m l? trung b?nh c?c dataset v? seed c?ng quy m?. Kh?ng c? ??nh tr?c ph? s?ng ? 97,5?100,5%, tr?nh che m?t k?t qu? th?p h?n.

In [ ]:
if tables is not None:
    labels = {"fitness_mean":"Fitness ?", "coverage":"Ph? s?ng ?", "energy":"N?ng l??ng chu?n h?a ?", "interference":"Nhi?u ?", "wall_s":"Th?i gian (s) ?", "fe":"S? FE"}
    for group, names in GROUPS.items():
        fig, axes = plt.subplots(2, 3, figsize=(16, 9))
        for ax, (metric, label) in zip(axes.flat, labels.items()):
            for name in names:
                g = tables["by_scale"].query("case == @name").sort_values("users")
                ax.plot(g.users, g[metric], marker="o", label=name)
            ax.set(xlabel="S? user", ylabel=label); ax.grid(alpha=.25)
        axes.flat[0].legend(fontsize=7); fig.suptitle(group); fig.tight_layout()
        fig.savefig(figure_dir / f"scale_{group}.png", dpi=180)
        plt.show()

## 6. Minh h?a nghi?m 3D
Ch?n tr??c dataset 200 user ??u ti?n theo t?n, seed ??u ti?n, I-WOA ? v?ng 100; kh?ng ch?n theo thu?t to?n th?ng. V? nghi?m ?? l?u, kh?ng t?i ?u l?i.

In [ ]:
if tables is not None:
    runs, history = read_results(OUTPUT)
    sample = runs.query("users == 200 and case == 'I-WOA'").sort_values(["dataset", "seed"]).iloc[0]
    import hashlib
    task_id = hashlib.sha256(f"{sample.dataset}|{sample.seed}|{sample['case']}".encode()).hexdigest()[:24]
    artifact = json.loads((OUTPUT / "tasks" / f"{task_id}.json").read_text(encoding="utf-8"))
    row = artifact["history"][100]
    import numpy as np
    from src.input.data_loader import load_users
    from src.preprocessing.preprocessor import preprocess_users
    users = preprocess_users(load_users(ROOT / "data/stores" / sample.dataset))
    uavs = np.asarray(row["position"]).reshape(-1, 3)
    fig = plt.figure(figsize=(10, 7)); ax = fig.add_subplot(111, projection="3d")
    ax.scatter(users.x_m, users.y_m, users.z_m, s=8, label="UE")
    ax.scatter(*uavs.T, marker="^", s=100, label="UAV")
    ax.set(xlabel="X (m)", ylabel="Y (m)", zlabel="Z (m)", title=f"{sample.dataset} | seed {sample.seed} | I-WOA")
    ax.legend(); fig.tight_layout(); fig.savefig(figure_dir / "placement_3d.png", dpi=180); plt.show()

## 7. ??nh gi? v? k?t lu?n
- ??c b?ng `ranking` (h?ng t? trung v? seed tr?n t?ng dataset), `contributions` v? `stability`.
- N?u ph??ng ?n c? fitness t?t, th?i gian th?p v? ?? ?n ??nh t?t; kh?ng m?c ??nh m?t ph??ng ?n t?t nh?t ? m?i quy m?.
- Ch?nh l?ch trong b?ng ??ng g?p l? so s?nh gh?p c?p b?t/t?t ??ng m?t th?nh ph?n.
- C?c b?ng hi?n l? m? t?; ch?a c? ki?m ??nh/CI cho 30 c?u h?nh. Kh?ng tuy?n b? kh?c bi?t c? ? ngh?a th?ng k?.
- C?ng 100 v?ng kh?ng ??ng ngh?a c?ng FE. Ghi r? chi ph? tr??c khi k?t lu?n.
- Ch? k?t lu?n tr?n b? d? li?u v? danh s?ch seed th?c s? ?? ch?y. Ch?a ch?y th? ch?a c? k?t lu?n.